In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install -U langchain langchain-community langchain-core transformers==4.52.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 82.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.9/136.9 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.5/247.5 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 104.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.5/160.5 kB 13.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Fou

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_core.prompts import PromptTemplate
import torch
import re

model_name = "mistralai/Mistral-Nemo-Instruct-2407"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")

def generate_text(prompt, max_length=1000, num_return_sequences=1):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7,
    )
    return [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00003-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

model-00005-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

model-00004-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [4]:
full_name_schema = ResponseSchema(
    name="full_name",
    description="The candidate's full name as a string."
)
email_schema = ResponseSchema(
    name="email",
    description="The candidate's email address as a string."
)
education_schema = ResponseSchema(
    name="education",
    description="A JSON array (as a string) of education entries, each with 'degree', 'institution', and 'year'."
)
skills_schema = ResponseSchema(
    name="skills",
    description="A JSON array (as a string) of the candidate's skills, e.g. ['Python', 'Machine Learning']."
)
experience_schema = ResponseSchema(
    name="experience",
    description="A JSON array (as a string) of experience entries, each with 'role', 'company', and 'years'."
)



In [5]:
response_schemas = [full_name_schema, email_schema, education_schema, skills_schema, experience_schema]
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = output_parser.get_format_instructions()

cv_extraction_template = """
You are a smart assistant that extracts a candidate's profile information from a CV / resume snippet.

Extract the full name, email, education history, skills, and work experience.

Respond ONLY in JSON format as follows:
{format_instructions}

Example Input:
\"Sara Ahmed - sara.ahmed@email.com
Education: M.Sc. Data Science, Cairo University, 2019
Skills: SQL, NLP, Deep Learning
Experience:
- Data Analyst at Amazon (2019-2021)
- ML Engineer at Microsoft (2021-Present)\"

Now extract from the following input:
\"{user_input}\"
"""

In [7]:
print (cv_extraction_template)


You are a smart assistant that extracts a candidate's profile information from a CV / resume snippet.

Extract the full name, email, education history, skills, and work experience.

Respond ONLY in JSON format as follows:
{format_instructions}

Example Input:
"Sara Ahmed - sara.ahmed@email.com
Education: M.Sc. Data Science, Cairo University, 2019
Skills: SQL, NLP, Deep Learning
Experience:
- Data Analyst at Amazon (2019-2021)
- ML Engineer at Microsoft (2021-Present)"

Now extract from the following input:
"{user_input}"



In [ ]:
user_input = """John Smith – john.smith@email.com
Education: B.Sc. Computer Science, MIT, 2020
Skills: Python, Machine Learning, Data Analysis
Experience:
- Software Engineer at Google (2020–2023)
- Data Scientist at OpenAI (2023–Present)"""


In [8]:

prompt = PromptTemplate(
    template=cv_extraction_template,
    input_variables=["user_input", "format_instructions"]
).format(user_input=user_input, format_instructions=format_instructions)

response = generate_text(prompt, max_length=1500)[0]
print(response)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



You are a smart assistant that extracts a candidate's profile information from a CV / resume snippet.

Extract the full name, email, education history, skills, and work experience.

Respond ONLY in JSON format as follows:
The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"full_name": string  // The candidate's full name as a string.
	"email": string  // The candidate's email address as a string.
	"education": string  // A JSON array (as a string) of education entries, each with 'degree', 'institution', and 'year'.
	"skills": string  // A JSON array (as a string) of the candidate's skills, e.g. ['Python', 'Machine Learning'].
	"experience": string  // A JSON array (as a string) of experience entries, each with 'role', 'company', and 'years'.
}
```

Example Input:
"Sara Ahmed - sara.ahmed@email.com
Education: M.Sc. Data Science, Cairo University, 2019
Skills: SQL, NLP, Deep Learning
Experie

In [9]:
def extract_json_block(text):
    pattern = r'```json\s*(.*?)\s*```'
    matches = re.findall(pattern, text, re.DOTALL)

    return f"```json\n{matches[-1]}\n```"

In [10]:
json_text = extract_json_block(response)
print(json_text)

```json
{
	"full_name": "John Smith",
	"email": "john.smith@email.com",
	"education": "[{\"degree\": \"B.Sc. Computer Science\", \"institution\": \"MIT\", \"year\": 2020}]",
	"skills": "[\"Python\", \"Machine Learning\", \"Data Analysis\"]",
	"experience": "[{\"role\": \"Software Engineer\", \"company\": \"Google\", \"years\": \"2020–2023\"}, {\"role\": \"Data Scientist\", \"company\": \"OpenAI\", \"years\": \"2023–Present\"}]"
}
```


In [11]:
output_data = output_parser.parse(json_text)
print(output_data)

{'full_name': 'John Smith', 'email': 'john.smith@email.com', 'education': '[{"degree": "B.Sc. Computer Science", "institution": "MIT", "year": 2020}]', 'skills': '["Python", "Machine Learning", "Data Analysis"]', 'experience': '[{"role": "Software Engineer", "company": "Google", "years": "2020–2023"}, {"role": "Data Scientist", "company": "OpenAI", "years": "2023–Present"}]'}
